In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# The published protocol describes 1,527 ordered records before IQR filtering:
# Operational assumption: reserve the last 500 source rows before target-based filtering.
# Confirm this row-order boundary against the source protocol before interpreting results.
df = pd.read_excel('../data/new_data_x_names.xlsx')
target_cols = ['x25', 'x26', 'x27', 'x28', 'x29']
if len(df) != 1527:
    raise ValueError('Expected 1,527 ordered records (1,027 training candidates + 500 holdout).')
if df[target_cols].isna().any().any():
    raise ValueError('Missing outcomes must be resolved from source records or excluded before this step; do not impute targets.')
train_candidates = df.iloc[:1027].copy()
df_holdout = df.iloc[1027:].copy()

# IQR bounds are estimated from training candidates only; test records are never filtered.
outlier_mask = pd.Series(False, index=train_candidates.index)
for col in target_cols:
    q1 = train_candidates[col].quantile(0.25)
    q3 = train_candidates[col].quantile(0.75)
    iqr = q3 - q1
    outlier_mask |= (train_candidates[col] < q1 - 1.5 * iqr) | (train_candidates[col] > q3 + 1.5 * iqr)

df_clean = train_candidates.loc[~outlier_mask].copy()
if len(df_clean) != 900:
    raise ValueError(f'Expected 900 training records after train-only IQR filtering; got {len(df_clean)}. Review the predeclared rule/data provenance.')
if len(df_holdout) != 500:
    raise ValueError('Holdout row count changed unexpectedly.')

# Descriptive plots use training candidates only.
fig, axes = plt.subplots(len(target_cols), 2, figsize=(15, 4 * len(target_cols)))
for i, col in enumerate(target_cols):
    sns.boxplot(x=train_candidates[col], ax=axes[i, 0])
    axes[i, 0].set_title(f'{col} - training candidates before filtering')
    sns.boxplot(x=df_clean[col], ax=axes[i, 1])
    axes[i, 1].set_title(f'{col} - training after filtering')
plt.tight_layout()
plt.show()

# Export training and holdout separately; do not inspect/filter holdout outcomes here.
ordered_clean = pd.concat([df_clean, df_holdout], axis=0)
ordered_clean.to_excel('../data/quyi_data_x_names.xlsx', index=False)
df_holdout.to_excel('../data/holdout_data.xlsx', index=False)
print(f'Training records retained: {len(df_clean)}; holdout retained unchanged: {len(df_holdout)}; canonical rows: {len(ordered_clean)}')